# Actividad 4 Parte 2

In [1]:

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
 
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, roc_auc_score, roc_curve)

df = pd.read_csv("../DataSets/Titanic.csv")
df.replace("?", np.nan, inplace=True)
 
# Corregir tipos numéricos que quedaron como object tras el replace
df["age"]  = pd.to_numeric(df["age"],  errors="coerce")
df["fare"] = pd.to_numeric(df["fare"], errors="coerce")
 
print(f"Filas cargadas : {len(df):,}")
print(f"Columnas       : {list(df.columns)}")
print("\nPrimeras 3 filas:")
display(df.head(3))
 

Filas cargadas : 1,309
Columnas       : ['pclass', 'survived', 'name', 'sex', 'age', 'sibsp', 'parch', 'ticket', 'fare', 'cabin', 'embarked', 'boat', 'body', 'home.dest']

Primeras 3 filas:


,pclass,survived,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home.dest
0,1,1,"Allen, Miss. Elisabeth Walton",female,29.0000,0,0,24160,211.3375,B5,S,2,NaN,"St Louis, MO"
1,1,1,"Allison, Master. Hudson Trevor",male,0.9167,1,2,113781,151.5500,C22 C26,S,11,NaN,"Montreal, PQ / Chesterville, ON"
2,1,0,"Allison, Miss. Helen Loraine",female,2.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"


## Limpieza de datos

In [2]:
cols_eliminar = ["name", "ticket", "cabin", "boat", "body", "home.dest"]
df.drop(columns=cols_eliminar, inplace=True)
print(f"Columnas eliminadas: {cols_eliminar}")
 
# Revisar nulos restantes
print("\nNulos por columna (tras eliminar cols. no útiles):")
print(df.isnull().sum())
 
# Eliminar filas con nulos
antes = len(df)
df.dropna(inplace=True)
print(f"\nFilas antes de dropna : {antes:,}")
print(f"Filas después          : {len(df):,}  (eliminadas: {antes - len(df):,})")

Columnas eliminadas: ['name', 'ticket', 'cabin', 'boat', 'body', 'home.dest']

Nulos por columna (tras eliminar cols. no útiles):
pclass        0
survived      0
sex           0
age         263
sibsp         0
parch         0
fare          1
embarked      2
dtype: int64

Filas antes de dropna : 1,309
Filas después          : 1,043  (eliminadas: 266)


## Conversion

In [3]:
df["pclass"]   = df["pclass"].astype("category")
df["survived"] = df["survived"].astype(int)
df["sex"]      = df["sex"].astype("category")
df["embarked"] = df["embarked"].astype("category")
 
print("Tipos de dato actualizados:")
print(df.dtypes)

Tipos de dato actualizados:
pclass      category
survived       int64
sex         category
age          float64
sibsp          int64
parch          int64
fare         float64
embarked    category
dtype: object


## Visualisasion

In [7]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle("Titanic – Relación entre variables y supervivencia",
             fontsize=15, fontweight="bold", y=1.01)
 
colores = ["#e74c3c", "#2ecc71"]
 
# 4a. Supervivencia por sexo
sns.countplot(data=df, x="sex", hue="survived", ax=axes[0, 0],
              palette=colores)
axes[0, 0].set_title("Supervivencia por Sexo")
axes[0, 0].set_xlabel("Sexo"); axes[0, 0].set_ylabel("Cantidad")
axes[0, 0].legend(["No sobrevivió", "Sobrevivió"], title="")
 
# 4b. Supervivencia por clase
sns.countplot(data=df, x="pclass", hue="survived", ax=axes[0, 1],
              palette=colores, order=[1, 2, 3])
axes[0, 1].set_title("Supervivencia por Clase")
axes[0, 1].set_xlabel("Clase"); axes[0, 1].set_ylabel("Cantidad")
axes[0, 1].legend(["No sobrevivió", "Sobrevivió"], title="")
 
# 4c. Distribución de edad por supervivencia
sns.histplot(data=df, x="age", hue="survived", ax=axes[0, 2],
             bins=25, palette=colores, alpha=0.7, kde=True)
axes[0, 2].set_title("Distribución de Edad")
axes[0, 2].set_xlabel("Edad"); axes[0, 2].set_ylabel("Frecuencia")
axes[0, 2].legend(["No sobrevivió", "Sobrevivió"], title="")
 
# 4d. Distribución de tarifa por supervivencia
sns.boxplot(data=df, x="survived", y="fare", ax=axes[1, 0],
            palette=colores)
axes[1, 0].set_title("Tarifa por Supervivencia")
axes[1, 0].set_xlabel("Sobrevivió (0=No, 1=Sí)")
axes[1, 0].set_ylabel("Tarifa (USD)")
axes[1, 0].set_ylim(0, 300)
 
# 4e. Supervivencia por puerto de embarque
sns.countplot(data=df, x="embarked", hue="survived", ax=axes[1, 1],
              palette=colores)
axes[1, 1].set_title("Supervivencia por Puerto (Embarked)")
axes[1, 1].set_xlabel("Puerto"); axes[1, 1].set_ylabel("Cantidad")
axes[1, 1].legend(["No sobrevivió", "Sobrevivió"], title="")
 
# 4f. Tasa de supervivencia por clase y sexo
tasa = df.groupby(["pclass", "sex"])["survived"].mean().reset_index()
tasa_pivot = tasa.pivot(index="pclass", columns="sex", values="survived")
tasa_pivot.plot(kind="bar", ax=axes[1, 2], color=["#3498db", "#e91e8c"],
                rot=0, edgecolor="white")
axes[1, 2].set_title("Tasa de Supervivencia por Clase y Sexo")
axes[1, 2].set_xlabel("Clase"); axes[1, 2].set_ylabel("Tasa de Supervivencia")
axes[1, 2].set_ylim(0, 1); axes[1, 2].legend(title="Sexo")
axes[1, 2].yaxis.set_major_formatter(
    plt.matplotlib.ticker.PercentFormatter(xmax=1))
 
plt.tight_layout()
plt.savefig("titanic_visualizacion.png", dpi=150, bbox_inches="tight")
plt.show()

C:\Users\USER\AppData\Local\Temp\ipykernel_30420\1463894731.py:29: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df, x="survived", y="fare", ax=axes[1, 0],
C:\Users\USER\AppData\Local\Temp\ipykernel_30420\1463894731.py:56: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


![Imagen](/Semana%205/Actividad%204/IMG/titanic_visualizacion.png)

In [9]:
grupos_var = [
    ("age",  "Edad"),
    ("fare", "Tarifa"),
    ("sibsp","Hermanos/Cónyuge"),
    ("parch","Padres/Hijos"),
]
 
for var, label in grupos_var:
    g0 = df.loc[df["survived"] == 0, var]
    g1 = df.loc[df["survived"] == 1, var]
    t, p = stats.ttest_ind(g0, g1, equal_var=False)
    sig = "SIGNIFICATIVO" if p < 0.05 else "No significativo"
    print(f"\n  Variable: {label} ({var})")
    print(f"    Media no-sobrevivió : {g0.mean():.2f}")
    print(f"    Media sobrevivió    : {g1.mean():.2f}")
    print(f"    t = {t:.3f}  |  p = {p:.4e}  → {sig}")


  Variable: Edad (age)
    Media no-sobrevivió : 30.50
    Media sobrevivió    : 28.82
    t = 1.829  |  p = 6.7736e-02  → No significativo

  Variable: Tarifa (fare)
    Media no-sobrevivió : 25.15
    Media sobrevivió    : 53.26
    t = -7.373  |  p = 5.9349e-13  → SIGNIFICATIVO

  Variable: Hermanos/Cónyuge (sibsp)
    Media no-sobrevivió : 0.51
    Media sobrevivió    : 0.49
    t = 0.394  |  p = 6.9384e-01  → No significativo

  Variable: Padres/Hijos (parch)
    Media no-sobrevivió : 0.34
    Media sobrevivió    : 0.54
    t = -3.786  |  p = 1.6255e-04  → SIGNIFICATIVO


In [10]:
le_sex      = LabelEncoder()   # female=0, male=1
le_embarked = LabelEncoder()
 
df_model = df.copy()
df_model["sex"]      = le_sex.fit_transform(df_model["sex"])
df_model["embarked"] = le_embarked.fit_transform(df_model["embarked"])
df_model["pclass"]   = df_model["pclass"].astype(int)
 
features = ["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]
X = df_model[features]
y = df_model["survived"]
 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
 
print(f"  Clases sex codificadas     : {dict(zip(le_sex.classes_, le_sex.transform(le_sex.classes_)))}")
print(f"  Clases embarked codificadas: {dict(zip(le_embarked.classes_, le_embarked.transform(le_embarked.classes_)))}")
print(f"\n  Entrenamiento : {X_train.shape[0]:,}  muestras")
print(f"  Prueba        : {X_test.shape[0]:,}  muestras")

  Clases sex codificadas     : {'female': np.int64(0), 'male': np.int64(1)}
  Clases embarked codificadas: {'C': np.int64(0), 'Q': np.int64(1), 'S': np.int64(2)}

  Entrenamiento : 834  muestras
  Prueba        : 209  muestras


## CREACIÓN DEL MODELO

In [12]:
modelo = LogisticRegression(max_iter=500, random_state=42)
modelo.fit(X_train, y_train)
 
y_pred       = modelo.predict(X_test)
y_pred_proba = modelo.predict_proba(X_test)[:, 1]
 
acc     = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)
 
print(f"\n  Exactitud (Accuracy) : {acc:.4f}  ({acc*100:.2f} %)")
print(f"  ROC-AUC              : {roc_auc:.4f}")
print("\n  Reporte de Clasificación:")
print(classification_report(y_test, y_pred,
                             target_names=["No sobrevivió", "Sobrevivió"]))
 


  Exactitud (Accuracy) : 0.7656  (76.56 %)
  ROC-AUC              : 0.8335

  Reporte de Clasificación:
               precision    recall  f1-score   support

No sobrevivió       0.77      0.86      0.81       124
   Sobrevivió       0.76      0.62      0.68        85

     accuracy                           0.77       209
    macro avg       0.76      0.74      0.75       209
 weighted avg       0.76      0.77      0.76       209



## Estimación de los coeficientes y los odds ratio

In [13]:
coef_df = pd.DataFrame({
    "Variable"  : features,
    "Coeficiente": modelo.coef_[0].round(4),
    "Odds Ratio" : np.exp(modelo.coef_[0]).round(4),
}).sort_values("Odds Ratio", ascending=False)
 
print("\n" + coef_df.to_string(index=False))


Variable  Coeficiente  Odds Ratio
   parch       0.0365      1.0372
    fare       0.0023      1.0023
     age      -0.0357      0.9649
embarked      -0.1953      0.8226
   sibsp      -0.3210      0.7254
  pclass      -0.9903      0.3715
     sex      -2.4944      0.0825


## Conclusion

- No es solo dinero → también estructura familiar afecta supervivencia
- Las mujeres tuvieron una tasa de supervivencia mucho mayor que los hombres.
- Los pasajeros de 1ª clase tuvieron la mayor supervivencia.
- el modelo predice bien quien vive pero no quien sobrevive
